Importamos librerias

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


Cargamos los data frames

In [ ]:
# Carga CSV
bank_df = pd.read_csv('../datos/bank-additional.csv', sep=',')


In [ ]:
# Carga Excel con varias hojas
customer_df_2012 = pd.read_excel('../datos/customer-details.xlsx', sheet_name='2012')
customer_df_2013 = pd.read_excel('../datos/customer-details.xlsx', sheet_name='2013')
customer_df_2014 = pd.read_excel('../datos/customer-details.xlsx', sheet_name='2014')

Exploramos el primer Dataframe

In [ ]:
bank_df.sample(15)

In [ ]:
bank_df.info()

In [ ]:
bank_df.describe()

In [ ]:
print(bank_df.dtypes.value_counts())

In [ ]:
bank_df.describe().round(2)

In [ ]:
# Ver cantidad de valores faltantes por columna
missing_values_bank = bank_df.isnull().sum()

# Mostrar solo columnas que tienen al menos un valor nulo
missing_values_bank = missing_values_bank[missing_values_bank > 0]

print("Valores faltantes por columna:")
print(missing_values_bank)


In [ ]:
porcentaje_nulos_bank = bank_df.isnull().mean().round(4) * 100
print(porcentaje_nulos_bank)


Quitamos la primera columna de "bank_df"

In [ ]:
# quitar columna innecesaria de indice
bank_df = bank_df.drop(bank_df.columns[0], axis=1)


Cambiamos el tipo valores a boleano por las columnas siguientes: 
'default', 'housing' y 'loan'

In [ ]:
# comprobamos si el dato es 0-1 boleano o numerico indicando una cantidad con la finalidad de transformar la columna
# Valores únicos en la columna 'housing'
print("Valores únicos en 'housing':", bank_df['housing'].unique())

# Valores únicos en la columna 'loan'
print("Valores únicos en 'loan':", bank_df['loan'].unique())


In [ ]:
# convertir las columnas 'default', 'housing' y 'loan' a tipo booleano
bank_df[['default', 'housing', 'loan']] = bank_df[['default', 'housing', 'loan']].astype(bool)
# Verificamos los tipos de datos después de la conversión
print(bank_df.dtypes[['default', 'housing', 'loan']])

Remplazamos valores nulos de la columna "age" por un promedio correspondiente al perfil del cliente segun criterio de empleo y educacion.

In [ ]:
# Agrupar por 'job' y 'education' y calcular la media de 'age'
tabla_promedios = bank_df.groupby(['job', 'education'])['age'].mean().round(0).reset_index()

# Mostrar la tabla resultante
print(tabla_promedios)

In [ ]:
# gestion de nulos de la columna age
# paso 1: Agrupamos por 'job' y 'education' y rellenamos los valores nulos en 'age' con la media del grupo
bank_df['age'] = bank_df.groupby(['job', 'education'])['age'].transform(lambda x: x.fillna(round(x.mean(), 0)))
# Paso 2: Imputar los que siguen siendo NaN (grupos sin edad registrada) con la media global
bank_df['age'] = bank_df['age'].fillna(round(bank_df['age'].mean(), 0))



In [ ]:
bank_df['age'].isnull().sum()  # Verificamos que no queden nulos en 'age'


Cambiamos el formato del valor de 'age' a entero: 

In [ ]:
#convertimos la columna age de formato float a int
bank_df['age'] = bank_df['age'].astype(int)




In [ ]:
# Verificamos los tipos de datos después de la conversión
bank_df.info()


Eliminacion de lineas sin valores de los atributos 'job', 'marital' ,'housing', 'loan' y 'date' que representan menos de 2,4% del data frame


In [ ]:

bank_df = bank_df.dropna(subset=['job', 'marital', 'housing', 'loan','date'])


Añadimos el valor "unknown" en la columna "education" por todos las valores faltantes

In [ ]:
bank_df['education'] = bank_df['education'].fillna('unknown')


Comprobamos el avance en gestion de nulos

In [ ]:
porcentaje_nulos_bank = bank_df.isnull().mean().round(4) * 100
print(porcentaje_nulos_bank)

En la columna "pdays" el valor 999 esta presente cuando el cliente nunca fue contactado y el calculo desde la ultima llamada no se puede hacer. Cambiaremos este valor por NaN para poder excluirlo de calculo posteriores (promedio ect..)

In [ ]:
bank_df.loc[bank_df['pdays'] == 999, 'pdays'] = pd.NA


Cambiamos el formato del valor de 'age' y 'pday' a entero: 

In [ ]:
# Convertimos las columnas a tipo entero que soporte NA (Int64)
bank_df['pdays'] = bank_df['pdays'].astype('Int64')
bank_df['age'] = bank_df['age'].astype('Int64')


In [ ]:
bank_df.info()

Seguimos con la transformacion de datos con objeto a float por las columnas siguientes:  
'cons.price.idx', 'cons.conf.idx' y 'euribor3m'  
Cambiamos todas las comas por un punto para evitar errores

In [ ]:
# Lista de columnas a transformar
columns_to_convert = ['cons.price.idx', 'cons.conf.idx', 'euribor3m']

# Reemplazar comas por puntos y convertir a float
for col in columns_to_convert:
    bank_df[col] = bank_df[col].astype(str).str.replace(',', '.')
    bank_df[col] = bank_df[col].astype(float)



In [ ]:
bank_df.info()

Se observa una relacione entre las columnas "cons.price.idx" y "cons.conf.idx", parece que los valores se corresponden.
vamos a confirmar la hipotesis y si se confirma rellenaremos los valores faltantes

In [ ]:
# Agrupar por 'cons.conf.idx' y contar los valores únicos de 'cons.price.idx'
unique_mapping = bank_df.groupby('cons.conf.idx')['cons.price.idx'].nunique()

# Verificar si todos los valores únicos son 1
is_unique_mapping = (unique_mapping == 1).all()

# Mostrar el resultado
if is_unique_mapping:
    print("Cada valor único en 'cons.conf.idx' tiene un único valor correspondiente en 'cons.price.idx'.")
else:
    print("Hay valores en 'cons.conf.idx' que tienen más de un valor correspondiente en 'cons.price.idx'.")



In [ ]:
# Crear un diccionario de mapeo desde 'cons.conf.idx' a 'cons.price.idx'
mapping = bank_df.dropna(subset=['cons.price.idx']).drop_duplicates(subset=['cons.conf.idx'])\
    .set_index('cons.conf.idx')['cons.price.idx'].to_dict()

# Rellenar los valores faltantes en 'cons.price.idx' usando el mapeo
bank_df['cons.price.idx'] = bank_df.apply(
    lambda row: mapping[row['cons.conf.idx']] if pd.isna(row['cons.price.idx']) else row['cons.price.idx'],
    axis=1)



In [ ]:
bank_df.info()

In [ ]:
porcentaje_nulos_bank = bank_df.isnull().mean().round(4) * 100
print(porcentaje_nulos_bank)

In [ ]:
bank_df.sample(15)

Cambiamos y uniformizamos el formato de la columna "nr.employed"

In [ ]:
# Reemplazar comas por puntos antes de convertir a float
bank_df['nr.employed'] = bank_df['nr.employed'].astype(str).str.replace(',', '.')
bank_df['nr.employed'] = bank_df['nr.employed'].astype(float)


In [ ]:
# Convertir a texto con 1 decimal, útil para visualización o comparación
bank_df['nr.employed'] = bank_df['nr.employed'].map(lambda x: f"{x:.1f}")
# Verificar el resultado
print(bank_df['nr.employed'].unique())

In [ ]:
# quitamos el punto de la columna nr.employed
bank_df['nr.employed'] = bank_df['nr.employed'].str.replace('.', '')
# Verificar el resultado
print(bank_df['nr.employed'].unique())
# Convertir a int
bank_df['nr.employed'] = bank_df['nr.employed'].astype(int) 



In [ ]:
bank_df.sample(15)


Renombramos la columna "y" como "subscribed", reemplazamos los valores de la columna y el tipo a boleano

In [ ]:
# cambio de nombre de la columna 'y'
bank_df.rename(columns={'y': 'subscribed'}, inplace=True)
# Reemplazar los valores de la columna 'subscribed' y convertir a booleano
bank_df['subscribed'] = bank_df['subscribed'].map({'yes': True, 'no': False})
# Verificar el resultado
bank_df.sample(15)

Cambiamos los valores de la columna "date" a un formato fecha dd/mm/aa y creamos columna separadas por cada componente de la fecha

In [ ]:
# de primero traducimos los valores en ingles a traves de un diccionario

meses_es_en = {
    "enero": "January", "febrero": "February", "marzo": "March", "abril": "April",
    "mayo": "May", "junio": "June", "julio": "July", "agosto": "August",
    "septiembre": "September", "octubre": "October", "noviembre": "November", "diciembre": "December"
}

# Reemplazar los nombres de los meses en español por inglés
for mes_es, mes_en in meses_es_en.items():
    bank_df['date'] = bank_df['date'].str.replace(mes_es, mes_en, regex=False)

bank_df['date'].head(10)

In [ ]:

# Conviertimos la columna 'date' a datetime usando el formato adecuado
bank_df['date'] = pd.to_datetime(bank_df['date'], errors='coerce')
bank_df['date'].head(5)

In [ ]:
bank_df['date'] = bank_df['date'].dt.date

In [ ]:
bank_df['date'].head(5)

In [ ]:

# Asegurarse de que la columna 'date' es de tipo datetime
bank_df['date'] = pd.to_datetime(bank_df['date'], errors='coerce')

# Crear las columnas separadas para mes y año
bank_df['contact_month'] = bank_df['date'].dt.month
bank_df['contact_year'] = bank_df['date'].dt.year


In [ ]:

# Asegurarse de que la columna 'date' es de tipo datetime
bank_df['date'] = pd.to_datetime(bank_df['date'], format='%d-%B-%Y', errors='coerce').dt.date


In [ ]:
bank_df.sample(5)

In [ ]:
bank_df.info()

Estudio de correlacion con la columna Euribor

In [ ]:
#tenenemos que selecionar solo las columnas numéricas antes de calcular la correlación
numeric_bank_df = bank_df.select_dtypes(include=['float', 'int']).columns


In [ ]:
# Estudiamos la correlacion de la columna 'euribor3m'
correlations_euribor = bank_df[numeric_bank_df].corr()['euribor3m'].drop('euribor3m')
print(correlations_euribor)


In [ ]:
# estudiamos la correlacione con la columna 'cons.price.idx'

correlations_cons_price = bank_df[numeric_bank_df].corr()['cons.price.idx'].drop('cons.price.idx')
print(correlations_euribor)


In [ ]:
sns.histplot(bank_df['euribor3m'].dropna(), kde=True)
plt.title('Distribución de euribor3m')
plt.show()

In [ ]:
#  Visualizar correlaciones relevantes
variables = ['euribor3m', 'emp.var.rate', 'nr.employed', 'cons.price.idx']
correlation_matrix = bank_df[variables].corr()

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlaciones con euribor3m')
plt.show()


Imputación de valores faltantes en la columna 'euribor3m' usando regresión lineal basada en 'emp.var.rate'

In [ ]:
# Calcular la media de euribor3m
EURIBOR_MEAN = bank_df['euribor3m'].mean()

# Calcular la media de emp.var.rate
EMP_RATE_MEAN = bank_df['emp.var.rate'].mean()

# Calcular la correlación entre euribor3m y emp.var.rate
CORRELATION = bank_df['euribor3m'].corr(bank_df['emp.var.rate'])

# Mostrar resultados
print(f"EURIBOR_MEAN = {EURIBOR_MEAN:.4f}")
print(f"EMP_RATE_MEAN = {EMP_RATE_MEAN:.4f}")
print(f"CORRELATION = {CORRELATION:.4f}")


In [ ]:
# PASO 1: Verificar valores faltantes
missing_count = bank_df['euribor3m'].isna().sum()
total_count = len(bank_df)
missing_pct = (missing_count / total_count) * 100

print(f"Estado inicial de euribor3m:")
print(f"- Total de registros: {total_count:,}")
print(f"- Valores faltantes: {missing_count:,} ({missing_pct:.1f}%)")

# PASO 2: Imputación por regresión
print(f"\nAplicando imputación por regresión...")

# Parámetros de regresión (de análisis previo)
EURIBOR_MEAN = 3.6185
EMP_RATE_MEAN = 0.0771
CORRELATION = 0.9724

# Fórmula: euribor3m = media + correlación * (emp.var.rate - media_emp)
print(f"Usando fórmula: euribor3m = {EURIBOR_MEAN} + {CORRELATION} * (emp.var.rate - {EMP_RATE_MEAN})")

# Aplicar imputación
bank_df['euribor3m'] = bank_df.apply(lambda row: 
    EURIBOR_MEAN + CORRELATION * (row['emp.var.rate'] - EMP_RATE_MEAN)
    if pd.isna(row['euribor3m']) else row['euribor3m'], axis=1)

# PASO 3: Validación
print(f"\n✅ Imputación completada")
print(f"- Valores imputados: {missing_count:,}")
print(f"- Valores faltantes finales: {bank_df['euribor3m'].isna().sum()}")
print(f"- Completitud del conjunto de datos: {(1 - bank_df['euribor3m'].isna().mean()) * 100:.1f}%")

# Mostrar estadísticas finales
final_stats = bank_df['euribor3m'].describe()
print(f"\nEstadísticas finales de euribor3m:")
print(f"- Conteo: {final_stats['count']:,.0f}")
print(f"- Media: {final_stats['mean']:.4f}")
print(f"- Desviación estándar: {final_stats['std']:.4f}")
print(f"- Mínimo: {final_stats['min']:.4f}")
print(f"- Máximo: {final_stats['max']:.4f}")


se uniformiza el formato de la columna euribor3m con maximo 3 decimales

In [ ]:
bank_df['euribor3m'] = bank_df['euribor3m'].round(3)


In [ ]:
bank_df['euribor3m'].sample(10)


Combinamos los 3 data frames de customer-details en una sola tabla  

In [ ]:
customer_df = pd.concat([customer_df_2012, customer_df_2013, customer_df_2014], ignore_index=True)
customer_df.sample(15)

Comprobamos la correspondencia de un identificardor unico entre los dos data frame par combinarlos

In [ ]:
customer_df['ID'].isin(bank_df['id_']).all()
customer_df['ID'].isin(bank_df['id_']).value_counts()





In [ ]:
customer_df.head(5)

In [ ]:
customer_df.describe().round(2)


In [ ]:
# Ver cantidad de valores faltantes por columna en el DataFrame combinado
missing_values_customer = customer_df.isnull().sum()
missing_values_customer

In [ ]:
porcentaje_nulos_customer = customer_df.isnull().mean().round(4) * 100   
porcentaje_nulos_customer

In [ ]:
# registros que esten en customer_df pero no en bank_df
customer_df[~customer_df['ID'].isin(bank_df['id_'])].sample(25)

Formateamos la columna "Dt_Customer" a fecha con formato 

In [ ]:
customer_df['Dt_Customer'] = customer_df['Dt_Customer'].dt.date


In [ ]:
customer_df.head(5)


quitamos el indice del DF customer_df

In [ ]:
#quitamos el indice del DF customer_df
customer_df = customer_df.reset_index(drop=True)    
customer_df.sample(5)

Combinamos los data frame de banco y cliente y lo llamamos "df_bank_cust"

In [ ]:
# Realiza el merge para obtener solo las entradas que tienen coincidencia en ambos DataFrames
df_bank_cust = pd.merge(
    bank_df, 
    customer_df, 
    left_on='id_', 
    right_on='ID', 
    how='inner'  # Esta opción conserva solo las coincidencias
)

# Resultado
print(f"Número de coincidencias encontradas: {df_bank_cust.shape[0]}")


In [ ]:
df_bank_cust.sample(15)

Quitamos las columnas id_ y ID ya que tienen el mismo valor y ya no traen valor añadido

In [ ]:
df_bank_cust = df_bank_cust.drop(columns=['id_'])
df_bank_cust = df_bank_cust.drop(columns=['ID'])
df_bank_cust.sample(15)


Quitamos el indice correspondiente a la parte customer_df dentro del data frame df_bank_cust

In [ ]:
#quitamos el indice correspondiente a la parte customer_df dentro del data frame df_bank_cust
df_bank_cust = df_bank_cust.drop(columns=['Unnamed: 0'])
df_bank_cust.sample(5)

Reseteamos el indice

In [ ]:

df_bank_cust = df_bank_cust.reset_index()
df_bank_cust = df_bank_cust.drop(columns=['index'])
df_bank_cust.head(15)

In [ ]:
porcentaje_nulos_bank_cust = df_bank_cust.isnull().mean().round(4) * 100
print(porcentaje_nulos_bank_cust)

In [ ]:
df_bank_cust.sample(15)

In [ ]:
df_bank_cust.info()


creamos un fichero csv del data frame final ne la carpeta "datos"

In [ ]:
df_bank_cust.to_csv('../datos/df_bank_cust.csv', index=False)